In [5]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

df = pd.read_parquet(BASE_DIR / "data" / "job_raw.csv")

df.head()

,Employee_ID,Job_Title,Industry,Country,Education_Level,Years_Experience,AI_Replacement_Risk,Future_Demand_Score,Remote_Work_Possibility,Average_Salary_USD,Required_Skills,Automation_Level,Job_Growth_2030,Work_Hours_Per_Week,Company_Size,AI_Tool_Usage,Performance_Score,Upskilling_Needed,Job_Satisfaction,Hiring_Trend_2026
0,AIJ-100000,Data Engineer,Healthcare,Japan,PhD,1,0.25,0.78,Yes,207392,"Python, Communication, Deep Learning",Low,3,37,Enterprise,High,2.08,Yes,3.86,Growing
1,AIJ-100001,Healthcare Analyst,Retail,UK,Bachelor,24,0.73,0.33,No,140785,"Deep Learning, Azure, Communication, TensorFlo...",Low,-5,42,Startup,Moderate,4.54,No,4.23,Growing
2,AIJ-100002,HR Specialist,Education,Canada,High School,21,0.80,0.69,Yes,124800,"Kubernetes, Cloud Computing, TensorFlow, SQL, ...",Medium,-5,57,Startup,Low,3.14,No,3.54,Stable
3,AIJ-100003,Cybersecurity Analyst,Government,UK,Bachelor,5,0.29,0.94,No,199878,"Excel, Kubernetes, Prompt Engineering, Leadership",Medium,7,59,Enterprise,High,3.67,No,4.37,Declining
4,AIJ-100004,Healthcare Analyst,Education,UAE,PhD,20,0.11,0.92,No,178682,"SQL, Leadership, TensorFlow, Cybersecurity",Low,6,34,Startup,High,3.68,No,3.99,Stable


In [8]:
dim_job = df[[
    "Job_Title", 
    "Automation_Level",
    "AI_Tool_Usage",
    "Upskilling_Needed"   
    ]].drop_duplicates().reset_index(drop=True)

    # Add Surrogate Key
dim_job["Job_ID"] = dim_job.index + 1

#Reorder Columns
dim_job = dim_job[[
    "Job_ID",
    "Job_Title", 
    "Automation_Level",
    "AI_Tool_Usage",
    "Upskilling_Needed"   
    ]]

dim_job.head()


,Job_ID,Job_Title,Automation_Level,AI_Tool_Usage,Upskilling_Needed
0,1,Data Engineer,Low,High,Yes
1,2,Healthcare Analyst,Low,Moderate,No
2,3,HR Specialist,Medium,Low,No
3,4,Cybersecurity Analyst,Medium,High,No
4,5,Healthcare Analyst,Low,High,No


In [11]:
dim_location = df[["Country"]].drop_duplicates().reset_index(drop=True)

dim_location["Location_ID"] = dim_location.index + 1

dim_location = dim_location[[
    "Location_ID",
    "Country"
    ]]

dim_location.head()

,Location_ID,Country
0,1,Japan
1,2,UK
2,3,Canada
3,4,UAE
4,5,Germany


In [12]:
dim_industry = df[[
    "Industry",
    "Company_Size"
]].drop_duplicates().reset_index(drop=True)

dim_industry["Industry_ID"] = dim_industry.index + 1

dim_industry = dim_industry[[
    "Industry_ID",
    "Industry",
    "Company_Size"
]]

dim_industry.head()

,Industry_ID,Industry,Company_Size
0,1,Healthcare,Enterprise
1,2,Retail,Startup
2,3,Education,Startup
3,4,Government,Enterprise
4,5,Energy,Medium


In [13]:
dim_education = df[["Education_Level",]].drop_duplicates().reset_index(drop=True)

dim_education["Education_ID"] = dim_education.index + 1

dim_education = dim_education[[
    "Education_ID",
    "Education_Level"
]]

dim_education.head()

,Education_ID,Education_Level
0,1,PhD
1,2,Bachelor
2,3,High School
3,4,Master


In [15]:
dim_worktype = df[["Remote_Work_Possibility"]].drop_duplicates().reset_index(drop=True)

dim_worktype["Worktype_ID"] = dim_worktype.index + 1

dim_worktype = dim_worktype[[
    "Worktype_ID",
    "Remote_Work_Possibility"
]]

dim_worktype.head()

,Worktype_ID,Remote_Work_Possibility
0,1,Yes
1,2,No
2,3,Hybrid


In [18]:
skills = df[["Employee_ID", "Required_Skills"]].copy()

skills = skills.assign(
    Skills = skills["Required_Skills"].str.split(",")
).explode("Required_Skills")

skills["Required_Skills"] = skills["Required_Skills"].str.strip()

skills.head()

,Employee_ID,Required_Skills,Skills
0,AIJ-100000,"Python, Communication, Deep Learning","[Python, Communication, Deep Learning]"
1,AIJ-100001,"Deep Learning, Azure, Communication, TensorFlo...","[Deep Learning, Azure, Communication, Tenso..."
2,AIJ-100002,"Kubernetes, Cloud Computing, TensorFlow, SQL, ...","[Kubernetes, Cloud Computing, TensorFlow, S..."
3,AIJ-100003,"Excel, Kubernetes, Prompt Engineering, Leadership","[Excel, Kubernetes, Prompt Engineering, Lea..."
4,AIJ-100004,"SQL, Leadership, TensorFlow, Cybersecurity","[SQL, Leadership, TensorFlow, Cybersecurity]"


In [20]:
dim_skill = skills[["Required_Skills"]].drop_duplicates().reset_index(drop=True)

dim_skill["Skill_ID"] = dim_skill.index + 1

dim_skill = dim_skill[[
    "Skill_ID",
    "Required_Skills"
]]

dim_skill.head()

,Skill_ID,Required_Skills
0,1,"Python, Communication, Deep Learning"
1,2,"Deep Learning, Azure, Communication, TensorFlo..."
2,3,"Kubernetes, Cloud Computing, TensorFlow, SQL, ..."
3,4,"Excel, Kubernetes, Prompt Engineering, Leadership"
4,5,"SQL, Leadership, TensorFlow, Cybersecurity"


In [22]:
bridge_employee_skill = skills.merge(
    dim_skill,
    on="Required_Skills",
    how="left"
)

bridge_employee_skill = bridge_employee_skill[[
    "Employee_ID",
    "Skill_ID"
]]

bridge_employee_skill.head()

,Employee_ID,Skill_ID
0,AIJ-100000,1
1,AIJ-100001,2
2,AIJ-100002,3
3,AIJ-100003,4
4,AIJ-100004,5


In [23]:
fact_jobs = df.copy()

In [25]:
fact_jobs = fact_jobs.merge(
    dim_job,
    on=[
        "Job_Title", 
        "Automation_Level",
        "AI_Tool_Usage",
        "Upskilling_Needed"   
    ],
    how="left"
)

In [26]:
fact_jobs = fact_jobs.merge(
    dim_location,
    on="Country",
    how="left"
)

In [27]:
fact_jobs = fact_jobs.merge(
    dim_industry,
    on=[
        "Industry",
        "Company_Size"
    ],
    how="left"
)

In [28]:
fact_jobs = fact_jobs.merge(
    dim_education,
    on="Education_Level",
    how="left"
)

In [30]:
fact_jobs = fact_jobs.merge(
    dim_worktype,
    on="Remote_Work_Possibility",
    how="left"
)

In [ ]:
fact_jobs = fact_jobs[[
    "Employee_ID",
    "Job_ID",
    "Location_ID",
    "Industry_ID",
    "Education_ID",
    "Worktype_ID",
    "Years_Experience",
    "AI_Replacement_Risk",
    "Future_Demand_Score",
    "Average_Salary_USD",
    "Job_Growth_2030",
    "Work_Hours_Per_Week",
    "Performance_Score",
    "Job_Satisfaction"
]]

fact_jobs.head()

KeyError: "['employee_ID', 'location_ID', 'Years_of_Experience', 'Job_Satisfaction_Score'] not in index"